In [ ]:
#@title Imports
# ============================================
# Cell 1
# Install + import dependencies (Google Colab)
# ============================================

!pip -q install opencv-python-headless pandas numpy tqdm

import math
from collections import deque

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

print("Environment ready.")

In [ ]:
#@title Geometry + Pose Helpers
# ============================================
# Cell 2
# Motion direction + pose classification helpers
# (ported from multiParams.py, sharing stentTrack.py's logic)
# ============================================


def motion_direction(prev_xy, curr_xy):
    """
    Compute direction of motion between two points.

    Based on angle between current and previous centroid coordinates.

    Returns:
        float: angle in degrees (NaN if undefined)
    """

    if prev_xy is None or np.any(np.isnan(curr_xy)):
        return np.nan
    dx = curr_xy[0] - prev_xy[0]
    dy = curr_xy[1] - prev_xy[1]
    if dx == 0 and dy == 0:
        return np.nan
    return math.degrees(math.atan2(dy, dx))


def unit(v):
    """
    Normalize a vector.
    """

    n = np.linalg.norm(v)
    return v / n if n > 0 else v


def angle_between(v1, v2):
    """
    Compute angle (radians) between two vectors.
    """
    v1 = unit(v1)
    v2 = unit(v2)
    dot = np.clip(np.dot(v1, v2), -1, 1)
    return math.acos(dot)


def determine_pose(contour, aspect_thresh=2.0):
    """
    Determine pose using longest contour axis and oriented bounding box.

    Returns:
        pose
        rect
        box
        aspect_ratio
        axis_data
    """

    pts = contour.reshape(-1, 2).astype(float)

    # ----------------------------------
    # Find longest distance pair
    # ----------------------------------

    max_dist = 0
    axis_p1 = None
    axis_p2 = None

    for i in range(len(pts)):
        d = np.linalg.norm(pts[i+1:] - pts[i], axis=1)

        if len(d):
            idx = np.argmax(d)

            if d[idx] > max_dist:
                max_dist = d[idx]
                axis_p1 = pts[i]
                axis_p2 = pts[i+1+idx]

    long_axis = axis_p2 - axis_p1
    long_axis = unit(long_axis)

    # perpendicular short axis
    short_axis = np.array([
        -long_axis[1],
         long_axis[0]
    ])

    # ----------------------------------
    # Project contour onto axes
    # ----------------------------------

    center = pts.mean(axis=0)

    proj_long = np.dot(pts-center, long_axis)
    proj_short = np.dot(pts-center, short_axis)

    long_min = proj_long.min()
    long_max = proj_long.max()

    short_min = proj_short.min()
    short_max = proj_short.max()

    long_len = long_max-long_min
    short_len = short_max-short_min

    aspect_ratio = long_len / (short_len + 1e-6)

    if aspect_ratio >= aspect_thresh:
        pose = "ELONGATED"
    else:
        pose = "CONTRACTED"

    # ----------------------------------
    # Generate oriented bounding box
    # ----------------------------------

    corners = np.array([
        center + long_axis*long_min + short_axis*short_min,
        center + long_axis*long_max + short_axis*short_min,
        center + long_axis*long_max + short_axis*short_max,
        center + long_axis*long_min + short_axis*short_max,
    ])

    axis_data = {
        "center": center,
        "long_axis": long_axis,
        "short_axis": short_axis,
        "long_min": long_min,
        "long_max": long_max,
        "short_min": short_min,
        "short_max": short_max
    }

    return pose, corners, aspect_ratio, axis_data


def find_head_tail(contour, axis_data):
    """
    Determine head and tail using contour width.

    The contour is divided across the midpoint of the long axis.
    The half with larger average distance from the long axis is the head.
    """

    pts = contour.reshape(-1,2).astype(float)

    center = axis_data["center"]
    long_axis = axis_data["long_axis"]
    short_axis = axis_data["short_axis"]

    long_mid = (
        axis_data["long_min"] +
        axis_data["long_max"]
    ) / 2

    # project points
    long_proj = np.dot(pts-center, long_axis)
    short_dist = np.abs(
        np.dot(pts-center, short_axis)
    )

    # split contour into halves

    head_half = short_dist[long_proj >= long_mid]
    tail_half = short_dist[long_proj < long_mid]

    if len(head_half)==0 or len(tail_half)==0:
        return None,None,None

    head_width = np.mean(head_half)
    tail_width = np.mean(tail_half)

    if head_width > tail_width:

        head_points = pts[long_proj >= long_mid]
        tail_points = pts[long_proj < long_mid]

        movement_order = "NORMAL"

    else:

        head_points = pts[long_proj < long_mid]
        tail_points = pts[long_proj >= long_mid]

        movement_order = "REVERSED"

    head_center = head_points.mean(axis=0)
    tail_center = tail_points.mean(axis=0)

    return head_center, tail_center, movement_order


def parse_contour(contour_str):
    """
    Parse a multiTest.py-style contour string "x1:y1;x2:y2;..." back into a
    cv2-compatible contour array of shape (N, 1, 2), int32.

    Returns None if the string is empty/NaN (e.g. a 'forced' detection with
    no real contour evidence).
    """
    if contour_str is None or (isinstance(contour_str, float) and np.isnan(contour_str)):
        return None
    contour_str = str(contour_str).strip()
    if contour_str == "":
        return None

    pts = []
    for pair in contour_str.split(";"):
        x_str, y_str = pair.split(":")
        pts.append([int(round(float(x_str))), int(round(float(y_str)))])

    if len(pts) == 0:
        return None

    return np.array(pts, dtype=np.int32).reshape(-1, 1, 2)

In [ ]:
#@title Per-Track Analysis
# ============================================
# Cell 3
# Per-track pose / head-tail / motion analysis
# (ported from multiParams.py; replays stentTrack.py's per-video logic
# independently for each track_id)
# ============================================


def analyze_track(track_df):
    """
    Apply stentTrack.py's pose / head-tail / motion-direction logic to a
    single track's frames, in ascending frame order.

    Args:
        track_df : DataFrame slice for one track_id, NOT necessarily sorted.

    Returns:
        DataFrame — same rows (sorted by frame), with pose, movement,
        direction_deg columns appended.
    """
    track_df = track_df.sort_values("FRAME").reset_index(drop=True)

    pos_history    = deque(maxlen=6)
    last_direction = np.nan

    poses      = []
    movements  = []
    directions = []

    for _, row in track_df.iterrows():
        cx, cy = row["POSITION_X"], row["POSITION_Y"]

        pose     = ""
        movement = ""
        direction = np.nan

        if not (np.isnan(cx) or np.isnan(cy)):
            pos_history.append((cx, cy))
            if len(pos_history) == pos_history.maxlen:
                direction      = motion_direction(pos_history[0], pos_history[-1])
                last_direction = direction
            else:
                direction = last_direction

        contour = parse_contour(row.get("contour_points"))

        # fitEllipse (inside determine_pose) needs at least 5 points
        if contour is not None and len(contour) >= 5:
            pose, box, aspect_ratio, axis_data = determine_pose(contour, aspect_thresh=1.5)

            if pose == "CONTRACTED":
                movement = "UNDEFINED"
            elif pose == "ELONGATED":
                head_center, tail_center, orientation = find_head_tail(contour, axis_data)

                if head_center is not None:

                    # compare biological orientation to motion direction

                    movement_vector = np.array([math.cos(math.radians(direction)), math.sin(math.radians(direction))])
                    head_vector = head_center - tail_center

                    if np.dot(head_vector, movement_vector) > 0:
                        movement = "FORWARD"
                    else:
                        movement = "BACKWARD"

        poses.append(pose)
        movements.append(movement)
        directions.append(direction)

    track_df["pose"]          = poses
    track_df["movement"]      = movements
    track_df["direction_deg"] = directions

    return track_df


def analyze_csv(input_csv, output_csv):
    """
    Read multiTest.py's output CSV, run per-track analysis, and write the
    result — same columns plus pose/movement/direction_deg — sorted by
    track_id (ascending) then frame (ascending) within each track.
    """
    df = pd.read_csv(input_csv, dtype={"contour_points": str})

    track_ids = sorted(df["TRACK_ID"].unique())

    analyzed_chunks = []
    for tid in tqdm(track_ids, desc="Analyzing tracks"):
        track_df = df[df["TRACK_ID"] == tid]
        analyzed_chunks.append(analyze_track(track_df))

    result = pd.concat(analyzed_chunks, ignore_index=True)
    result.to_csv(output_csv, index=False)
    print(f"Saved analyzed CSV → {output_csv}")

In [ ]:
# ============================================
# Cell 4
# Notebook inputs + run
# ============================================

# -----------------------------
# Input / output files
# -----------------------------
# INPUT_CSV should be a tracks CSV produced by stentorDetect2.ipynb's
# export_tracks_csv (columns: TRACK_ID, FRAME, POSITION_X, POSITION_Y,
# contour_points).

INPUT_CSV  = "/content/Uranus_cam_test-2.csv"
OUTPUT_CSV = "/content/Uranus_cam_test-2_analyzed.csv"

# -----------------------------
# Run
# -----------------------------

analyze_csv(
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_CSV
)